In [ ]:
%%sql -r dataframe_1
USE DATABASE FINANCE_DB;
USE SCHEMA FINANCE;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import StructType, StructField, StringType

session = get_active_session()
session.use_role('ACCOUNTADMIN')
session.use_database('FINANCE_DB')
session.use_schema('FINANCE')

In [ ]:
schema = StructType([
    StructField('TRANSACTION_ID', StringType()),
    StructField('DATE', StringType()),
    StructField('TIME', StringType()),
    StructField('TYPE', StringType()),
    StructField('NAME', StringType()),
    StructField('EMOJI', StringType()),
    StructField('CATEGORY', StringType()),
    StructField('AMOUNT', StringType()),
    StructField('CURRENCY', StringType()),
    StructField('LOCAL_AMOUNT', StringType()),
    StructField('LOCAL_CURRENCY', StringType()),
    StructField('NOTES_AND_TAGS', StringType()),
    StructField('ADDRESS', StringType()),
    StructField('RECEIPT', StringType()),
    StructField('DESCRIPTION', StringType()),
])

df = session.read.schema(schema).option('skip_header', 1).option('field_optionally_enclosed_by', '"').csv('@finance_sources/Monzo/')
df.show(15)

In [ ]:
table_name = 'FINANCE_DB.FINANCE.MONZO_TRANSACTIONS'

existing_tables = [row['name'] for row in session.sql("SHOW TABLES IN FINANCE_DB.FINANCE").collect()]

if 'MONZO_TRANSACTIONS' not in existing_tables:
    df.write.mode('overwrite').save_as_table(table_name)
    print(f'Created table and saved {df.count()} rows to {table_name}')
else:
    existing_df = session.table(table_name)
    new_rows = df.join(existing_df, on='TRANSACTION_ID', how='left_anti')
    count = new_rows.count()
    if count > 0:
        new_rows.write.mode('append').save_as_table(table_name)
        print(f'Appended {count} new rows to {table_name}')
    else:
        print(f'No new rows to append — all rows already exist in {table_name}')

In [ ]:
%%sql -r dataframe_3
--CREATE OR REPLACE VIEW FINANCE_DB.FINANCE.MONZO_TRANSACTIONS_VW AS
SELECT
    TRANSACTION_ID,
    TO_DATE(DATE, 'DD/MM/YYYY') AS DATE,
    TYPE,
    NAME,
    CATEGORY,
    AMOUNT::NUMBER(10,2) AS AMOUNT,
    CURRENCY,
    NOTES_AND_TAGS,
    ADDRESS,
    DESCRIPTION
FROM FINANCE_DB.FINANCE.MONZO_TRANSACTIONS
LIMIT 10

In [ ]:
%%sql -r dataframe_2
SELECT DATE_TRUNC('month', TO_DATE(DATE, 'DD/MM/YYYY')) AS MONTH, 
        count(1) as TRAN_CNT,
        SUM(TRY_CAST(LOCAL_AMOUNT AS NUMBER(10,2))) AS TOTAL_LOCAL_AMOUNT
FROM FINANCE_DB.FINANCE.MONZO_TRANSACTIONS
GROUP BY MONTH
ORDER BY MONTH;